# SHA-1 安全散列算法

**摘要：** SHA-1是160位哈希算法，曾广泛应用，后因碰撞漏洞被SHA-2/3取代

- **类别：** 现代密码学
- **难度：** 中等
- **预计用时：** 3小时
- **先修：** 基础哈希函数概念、二进制运算、模运算
- **学习目标：** 掌握SHA-1原理与实现

> 说明：本课程强调“可运行 + 可解释 + 可练习”。代码尽量仅使用 Python 标准库（Pyodide 友好）。

## 你将获得什么

- **固定输出 160位：** 输出长度固定为160比特
- **迭代压缩  Merkle–Damgård：** 采用迭代压缩函数结构
- **已被攻破 无安全性：** 存在碰撞漏洞已被淘汰

## 学习路线图（从直觉到实现）

我们把学习过程拆成 4 层，每一层都尽量给出“可验证”的产物：

1. **直觉层**：能说清楚它解决什么安全目标，以及为什么需要“密钥/参数/随机性”。
2. **符号层**：能把关键变换写成一个简短公式，例如 $y=f(x,k)$ 或 $$y=f(x,k)\bmod n$$。
3. **实现层**：能写出可运行的函数/类，并通过至少 3 条 `assert` 自测。
4. **对抗层**：能指出它可能被怎么攻（至少一个思路），并用代码做一个最小实验验证。

> 你会发现：能通过“对抗层”的小实验，往往才算真正理解。


## 应用场景与安全性讨论（扩充阅读）

这一主题通常会在以下场景出现（不同主题侧重点不同）：

- **教学/入门**：用简化模型理解“密钥 + 变换”的思想。
- **工程系统**：用成熟算法与标准协议实现保密性/完整性/认证。
- **攻防分析**：通过已知攻击理解“为什么某些方案不再推荐”。

### 安全目标（Security Goals）

常见目标包括：

- **保密性（Confidentiality）**：未授权者无法获得明文信息。
- **完整性（Integrity）**：消息在传输中被篡改能被发现。
- **认证（Authentication）**：确认对方是谁（或确认数据来自谁）。
- **不可否认（Non-repudiation）**：事后不能否认曾经生成/发送过某信息（通常与签名相关）。

### 威胁模型（Threat Model）

做任何“安全性结论”之前，先明确攻击者能做什么：

- 只能看到密文？还是还能选择明文（chosen-plaintext）？
- 能否篡改、重放、插入消息？
- 能否获取部分密钥/随机数？是否存在侧信道？

> 调试提示：如果你觉得“算法没问题但结果不对”，先检查你实现的威胁模型是否一致——例如你是否在复用 nonce/IV，或者把编码/填充当成了算法的一部分。


## 环境准备

In [ ]:
# 课程通用导入（尽量只用标准库）
import math
import random
import string
import secrets
import hashlib
import hmac
import itertools
import statistics
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Iterable

random.seed(0)  # 为了让演示更可复现


## 热身：数据与表示

很多实现问题来自“数据表示”而不是算法本身。请确保你能区分：

- 文本：`str`
- 字节：`bytes`
- 整数：`int`

并能相互转换。

In [ ]:
msg = "Hello, 密码学"
b = msg.encode("utf-8")
print(type(msg), type(b))  # 预期输出：<class 'str'> <class 'bytes'>
print(b[:10])              # 预期输出：前10个字节（与编码有关）
print(b.hex()[:20])        # 预期输出：十六进制字符串前20个字符


### 工具函数：字节/整数/十六进制

In [ ]:
def bytes_to_hex(b: bytes) -> str:
    return b.hex()

def hex_to_bytes(h: str) -> bytes:
    h = h.strip().lower()
    if h.startswith("0x"):
        h = h[2:]
    return bytes.fromhex(h)

def int_to_bytes(x: int, length: int, byteorder: str = "big") -> bytes:
    return x.to_bytes(length, byteorder=byteorder, signed=False)

def bytes_to_int(b: bytes, byteorder: str = "big") -> int:
    return int.from_bytes(b, byteorder=byteorder, signed=False)

assert hex_to_bytes(bytes_to_hex(b"ABC")) == b"ABC"


## 核心内容分步讲解

### Step 1: 了解SHA-1的起源与背景

1. SHA-1全称为Secure Hash Algorithm（安全散列算法），由美国国家安全局（NSA）设计，美国国家标准与技术研究院（NIST）于1995年发布。
2. 它是SHA家族第二个成员，前身SHA-0因安全漏洞被快速弃用，SHA-1是其改进版本，核心差异在于压缩函数的消息扩展步骤，新增了消息调度的循环移位操作以提升安全性。
3. SHA-1的设计深受MD4/MD5哈希函数影响，目标是保证抗碰撞性、抗原像性和次原像性，最初用于数字签名和完整性校验等场景。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 2: 梳理SHA-1的应用历史

1. 标准化阶段：1995年被收录到FIPS PUB 180-1联邦信息处理标准，成为美国联邦机构和商业系统的标配哈希算法。
2. 广泛应用阶段：覆盖SSL/TLS加密通信、PGP/SSH/IPsec安全协议、数字签名、软件/文件完整性校验等核心安全场景。
3. 淘汰阶段：2005年王小云团队公布碰撞攻击方法，2017年Google与CWI发布首个实际碰撞实例（SHAttered攻击），主流浏览器禁用SHA-1证书，NIST将其标记为不推荐使用，逐步被SHA-2、SHA-3取代。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 3: 掌握SHA-1的数学结构

SHA-1核心采用Merkle–Damgård结构，整体流程分为三大核心环节：
1. 对输入消息进行预处理后分块，每个块大小为512比特；
2. 每个分块通过压缩函数与前序哈希状态迭代计算；
3. 所有分块处理完成后，输出最终的160比特哈希摘要。
该结构是哈希函数经典设计范式，保证了算法的迭代可计算性。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 4: 执行消息预处理操作

预处理是SHA-1保证输入唯一性的关键步骤，分为填充和分块两步：
1. 填充规则：
   - 设原始消息比特长度为$L$，先在消息末尾添加1比特的1；
   - 补充若干个0比特，直到消息总长度 ≡ 448 (mod 512)；
   - 最后附加$L$的64位二进制表示，使填充后总长度为512的整数倍。
2. 分块操作：将填充后的完整消息拆分为若干个512比特的分组，记为$M^{(1)}, M^{(2)}, …, M^{(N)}$。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 5: 初始化哈希向量（IV）

SHA-1的内部状态由5个32位寄存器组成，初始值固定为：
$H_0 = 0x67452301$
$H_1 = 0xEFCDAB89$
$H_2 = 0x98BADCFE$
$H_3 = 0x10325476$
$H_4 = 0xC3D2E1F0$
这5个初始值是算法预置的固定常量，作为迭代计算的初始哈希状态。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 6: 执行迭代压缩函数计算

对每个512比特分组$M^{(i)}$，执行以下核心计算：
1. 消息调度：
   - 将分组拆分为16个32位字：$W_0, W_1, …, W_{15}$；
   - 扩展为80个32位字：$W_t = ROL_1(W_{t-3} ⊕ W_{t-8} ⊕ W_{t-14} ⊕ W_{t-16})$（$t=16..79$），其中$ROL_1$表示32位整数左循环移位1位。
2. 80轮迭代运算：
   - 初始化工作寄存器：$(A,B,C,D,E) = (H_0,H_1,H_2,H_3,H_4)$；
   - 每轮计算：$TEMP = ROL_5(A) + f_t(B,C,D) + E + W_t + K_t$，随后更新寄存器：$E=D, D=C, C=ROL_{30}(B), B=A, A=TEMP$；
   - 轮函数$f_t$和常数$K_t$随轮次变化（0-19/20-39/40-59/60-79四组取值）。
3. 更新哈希值：处理完分组后，$H_0=H_0+A, H_1=H_1+B, H_2=H_2+C, H_3=H_3+D, H_4=H_4+E$（均为模$2^{32}$加法）。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 7: 生成最终SHA-1哈希输出

当所有512比特分组处理完成后，将最终的5个32位寄存器值拼接，得到160位的哈希摘要：
$SHA-1(M) = H_0 ∥ H_1 ∥ H_2 ∥ H_3 ∥ H_4$
其中$∥$表示比特串拼接，最终输出的160位结果即为输入消息的SHA-1哈希值，通常转换为40位十六进制字符串展示。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 8: 实现并验证哈希算法

基于SHA系列算法的核心逻辑，可通过Python实现哈希计算（注：示例为SHA-256实现，SHA-1可类比实现）：
1. 实现核心工具函数：如32位整数循环右移函数right_rotate；
2. 消息预处理：完成比特填充和分块操作；
3. 消息调度扩展：将512比特块扩展为64个32位字（SHA-256）/80个32位字（SHA-1）；
4. 迭代压缩计算：按轮次执行压缩函数，更新哈希状态；
5. 结果输出：将最终哈希状态转换为十六进制字符串。
6. 验证：使用标准测试向量（空字符串、"abc"等）验证实现正确性，确保输出与预期哈希值一致。


> 提示：注意区分SHA-1与SHA-256的参数差异（如初始向量、轮常数、轮次、输出长度等）

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
def right_rotate(value, amount):
    """32位整数的循环右移"""
    return ((value >> amount) | (value << (32 - amount))) & 0xFFFFFFFF


def sha256(message):
    """
    计算输入消息的SHA-256散列值
    
    参数:
        message (str 或 bytes): 要计算散列值的消息
    
    返回:
        str: SHA-256散列值的十六进制字符串表示
    """
    # 将输入消息转换为字节
    if isinstance(message, str):
        message = message.encode('utf-8')
    
    # SHA-256算法使用的初始哈希值 (前8个质数的平方根小数部分的前32位)
    h = [
        0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
        0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19
    ]
    
    # SHA-256算法使用的常量 (前64个质数的立方根小数部分的前32位)
    k = [
        0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
        0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
        0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
        0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
        0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
        0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
        0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
        0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
        0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
        0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
        0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
        0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
        0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
        0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
        0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
        0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2
    ]
    
    # 预处理: 填充消息
    # 1. 附加比特'1'
    # 2. 附加k个比特'0', k是满足 (消息长度 + 1 + k + 8) % 512 = 0 的最小非负整数
    # 3. 附加消息长度的64位表示
    
    # 计算原始比特长度
    original_bit_len = len(message) * 8
    
    # 附加比特'1'
    message += b'\x80'
    
    # 附加比特'0'直到消息长度满足条件 (消息长度 + 1 + k + 8) % 64 = 0 (以字节为单位)
    while (len(message) % 64) != 56:
        message += b'\x00'
    
    # 附加消息长度(原始消息的比特长度)的64位(8字节)表示
    message += original_bit_len.to_bytes(8, byteorder='big')
    
    # 处理消息块
    for i in range(0, len(message), 64):
        chunk = message[i:i+64]
        
        # 创建消息调度表 (message schedule)
        w = [0] * 64
        
        # 将块拆分为16个32位(4字节)字
        for j in range(16):
            w[j] = int.from_bytes(chunk[j*4:j*4+4], byteorder='big')
        
        # 扩展剩余的48个字
        for j in range(16, 64):
            s0 = right_rotate(w[j-15], 7) ^ right_rotate(w[j-15], 18) ^ (w[j-15] >> 3)
            s1 = right_rotate(w[j-2], 17) ^ right_rotate(w[j-2], 19) ^ (w[j-2] >> 10)
            w[j] = (w[j-16] + s0 + w[j-7] + s1) & 0xFFFFFFFF
        
        # 初始化工作变量为当前的哈希值
        a, b, c, d, e, f, g, h_val = h
        
        # 压缩函数主循环
        for j in range(64):
            S1 = right_rotate(e, 6) ^ right_rotate(e, 11) ^ right_rotate(e, 25)
            ch = (e & f) ^ ((~e) & g)
            temp1 = (h_val + S1 + ch + k[j] + w[j]) & 0xFFFFFFFF
            S0 = right_rotate(a, 2) ^ right_rotate(a, 13) ^ right_rotate(a, 22)
            maj = (a & b) ^ (a & c) ^ (b & c)
            temp2 = (S0 + maj) & 0xFFFFFFFF
            
            h_val = g
            g = f
            f = e
            e = (d + temp1) & 0xFFFFFFFF
            d = c
            c = b
            b = a
            a = (temp1 + temp2) & 0xFFFFFFFF
        
        # 将压缩后的块添加到当前哈希值
        h[0] = (h[0] + a) & 0xFFFFFFFF
        h[1] = (h[1] + b) & 0xFFFFFFFF
        h[2] = (h[2] + c) & 0xFFFFFFFF
        h[3] = (h[3] + d) & 0xFFFFFFFF
        h[4] = (h[4] + e) & 0xFFFFFFFF
        h[5] = (h[5] + f) & 0xFFFFFFFF
        h[6] = (h[6] + g) & 0xFFFFFFFF
        h[7] = (h[7] + h_val) & 0xFFFFFFFF
    
    # 将最终哈希值转换为十六进制字符串
    return ''.join(format(h_val, '08x') for h_val in h)

# 演示验证
test_vectors = [
    ("", "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"),
    ("abc", "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"),
    ("abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq",
     "248d6a61d20638b8e5c026930c3e6039a33ce45964ff2167f6ecedd419db06c1"),
]

for message, expected in test_vectors:
    result = sha256(message)
    print(f"消息: {message!r}")
    print(f"SHA-256: {result}")
    print(f"预期结果: {expected}")
    print(f"验证: {'通过' if result == expected else '失败'}")
    print()

# 预期输出：根据输入得到对应结果（请与讲解示例对照）

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

## 综合示例：端到端流程 + 自测

In [ ]:
# 端到端模板：将主题核心操作封装成接口
# 你可以把 encrypt/decrypt 或 hash/sign/verify 等组合在一起

def pipeline(data: bytes) -> bytes:
    # TODO: 替换为你的端到端流程
    return hashlib.sha256(data).digest()

def check_pipeline():
    a = pipeline(b"hello")
    b = pipeline(b"hello")
    c = pipeline(b"hellp")
    assert a == b
    assert a != c
    print("端到端自测通过")  # 预期输出：端到端自测通过

check_pipeline()


## 小实验：敏感性（雪崩效应）

In [ ]:
def hamming_distance_bytes(a: bytes, b: bytes) -> int:
    if len(a) != len(b):
        raise ValueError("长度必须相同才能计算差异度")
    return sum(x != y for x, y in zip(a, b))

def flip_one_bit(data: bytes, bit_index: int = 0) -> bytes:
    if not data:
        return data
    byte_i = bit_index // 8
    bit_i = bit_index % 8
    byte_i = byte_i % len(data)
    mask = 1 << bit_i
    arr = bytearray(data)
    arr[byte_i] ^= mask
    return bytes(arr)

def core_transform(data: bytes) -> bytes:
    # TODO: 替换成你的核心变换
    # 示例：用 SHA-256 作为“对照组”
    return hashlib.sha256(data).digest()

base = b"hello world"
out1 = core_transform(base)
out2 = core_transform(flip_one_bit(base, 0))

print("输出长度(字节):", len(out1))  # 预期输出：32（若使用SHA-256）
print("差异度(字节):", hamming_distance_bytes(out1, out2))  # 预期输出：通常 > 10


## 附录A：最常用的数学与位运算背景

### 1. 模运算（Modular Arithmetic）

当我们写 $$a \equiv b \pmod n$$ 时，意思是 $a$ 与 $b$ 除以 $n$ 的余数相同，也就是 $n$ 整除 $a-b$。

常见等价写法：

- $a \bmod n$：把 $a$ 映射到 $0..n-1$ 的代表元
- 若 $a<0$，Python 仍保证 $a \bmod n \in [0,n-1]$

**一个很重要的直觉：** 模运算会“折叠”整数轴，把无限多的整数映射到有限集合，所以很多密码体制会用到它来构造“封闭”的运算空间。

### 2. 最大公因数与互质

若 $$\gcd(a,n)=1$$，我们称 $a$ 与 $n$ 互质（coprime）。互质非常重要，因为它意味着 $a$ 在模 $n$ 意义下存在乘法逆元 $a^{-1}$，满足：

$$a\cdot a^{-1} \equiv 1 \pmod n$$

### 3. 扩展欧几里得算法（Extended Euclid）

它不仅能算 $\gcd(a,b)$，还能找到整数 $x,y$ 使得：

$$ax+by=\gcd(a,b)$$

当 $\gcd(a,n)=1$ 时，$x$ 就是 $a$ 关于模 $n$ 的逆元（可能为负，需要再取模）。

### 4. 异或（XOR）

在字节层面，异或常写成 $c=a\oplus b$，它有几个极其重要的性质：

- $a\oplus a=0$
- $a\oplus 0=a$
- $(a\oplus b)\oplus b=a$（可逆性）

因此很多流密码/分组模式会用异或把“密钥流”与明文混合。

> 调试提示：如果你发现解密结果不等于明文，先检查“同一份密钥流是否被一致地使用”，以及字节对齐是否正确。


In [ ]:
# 附录A配套代码：gcd、逆元、异或操作的最小实现与自测

def egcd(a: int, b: int) -> Tuple[int, int, int]:
    """返回 (g, x, y) 使得 a*x + b*y = g = gcd(a,b)"""
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = egcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return (g, x, y)

def modinv(a: int, n: int) -> int:
    g, x, _ = egcd(a, n)
    if g != 1:
        raise ValueError("不存在逆元：a 与 n 不互质")
    return x % n

print(math.gcd(3, 26))     # 预期输出：1
print(modinv(3, 26))       # 预期输出：9，因为 3*9=27≡1(mod26)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    if len(a) != len(b):
        raise ValueError("xor 需要等长字节串")
    return bytes(x ^ y for x, y in zip(a, b))

p = b"ABC"
k = b"\x01\x01\x01"
c = xor_bytes(p, k)
p2 = xor_bytes(c, k)
print(c)   # 预期输出：b'@BA'（因为 0x41^1=0x40 等）
print(p2)  # 预期输出：b'ABC'


## 常见错误 / 踩坑点 / 调试提示

> 1. **编码问题**：`str`/`bytes` 混用导致哈希/加解密结果不一致。
>
> 2. **参数合法性**：例如需要互质、需要固定长度、需要随机数不可复用。
>
> 3. **端序与位序**：大端/小端混淆、位操作方向写反。
>
> 4. **测试不足**：缺少边界与异常路径测试。
>
> 5. **把演示当安全方案**：课程中的简化实现不等价于生产安全实现。

## 练习题（含更详细要求）

### 练习1：功能完善（必做）
- 目标：把你的核心函数做成“更好用”的版本  
- 要求：
  - 输入支持 `str` 与 `bytes`
  - 对非法参数给出清晰报错（`ValueError`）
  - 至少写 5 条 `assert`（覆盖正常/边界/异常）

### 练习2：最小攻防实验（推荐）
- 目标：实现一个最小的攻击/对抗脚本  
- 示例方向（任选其一）：
  - 暴力枚举 key space
  - 频率分析/统计偏差
  - 重放/篡改检测失败示例
  - 碰撞/相同摘要的构造（仅演示，不鼓励用于真实攻击）

### 练习3：写一段“工程化清单”（可选）
- 目标：把课程知识迁移到真实系统  
- 建议包含：
  - 参数选择（key 长度、随机数、模式）
  - 依赖库与实现来源（优先标准/权威实现）
  - 安全审计点（日志、异常、边界）


In [ ]:
# 练习参考答案框架（示例）
# 说明：这里只提供“结构”，你需要填入你自己的实现。

def solve_ex1(data):
    if isinstance(data, str):
        data_b = data.encode("utf-8")
    elif isinstance(data, (bytes, bytearray)):
        data_b = bytes(data)
    else:
        raise ValueError("只支持 str/bytes")
    return hashlib.sha256(data_b).hexdigest()

assert solve_ex1("a") == solve_ex1("a")
assert solve_ex1("a") != solve_ex1("b")
try:
    solve_ex1(123)
except ValueError:
    pass

print("练习1框架可运行")  # 预期输出：练习1框架可运行


## 小结

把今天的内容压缩成 3 句话：

1. 这个主题的核心变换是 $y=f(x,k)$（或 $$y=f(x,k)\bmod n$$）。
2. 正确实现需要重视：数据表示、参数合法性、测试向量与边界条件。
3. 真正理解来自：能写出最小攻防实验，并解释现象原因。


## 随堂小测（10题）
1. 这个主题的“密钥/参数”是什么？它决定了哪些行为？
2. 为什么需要模运算/置换/异或等操作？分别带来什么效果？
3. 在你的实现里，哪一步最容易写错？你如何用测试发现它？
4. 若攻击者拥有密文（或摘要），最直接的攻击方式是什么？
5. 在什么场景下“能解密”不等于“安全”？举一个例子。
6. 是否存在“重复使用随机数/nonce/IV”的风险？为什么危险？
7. 如果输入非常长，你的实现是否会变慢？瓶颈在哪？
8. 你能写出一个最小“对照实验”，让输出变化更直观吗？
9. 如果要用于真实系统，你会替换/增强哪一部分？
10. 用一句话总结：你学到的最重要概念是什么？

> 自评：能回答 7/10 且能跑通练习，就算达标。
